<a href="https://colab.research.google.com/github/UZUddin/PitLane-Explained/blob/main/pitlane_explained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# PitLane Explained — IBM SkillsBuild AI Builders Challenge
# AI Race Companion for Casual F1 Fans
# Tools: IBM Granite, Docling, LangChain, ChromaDB

In [2]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    "transformers==4.49.0" \
    pillow \
    langchain_classic \
    langchain_core \
    langchain_huggingface sentence_transformers \
    langchain_chroma chromadb \
    docling \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git"
! echo "::endgroup::"

::group::Install Dependencies
Using Python 3.12.13 environment at: /usr
Resolved 162 packages in 1.07s
Prepared 1 package in 1.25s
Uninstalled 1 package in 270ms
Installed 1 package in 131ms
 - transformers==4.47.0
 + transformers==4.49.0
::endgroup::


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

embeddings_model_path = "ibm-granite/granite-embedding-small-english-r2"
embeddings_model = HuggingFaceEmbeddings(model_name=embeddings_model_path)
embeddings_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_path)

print("✅ Embeddings model ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/95.3M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embeddings model ready!


In [4]:
import os
from google.colab import userdata
from ibm_granite_community.notebook_utils import get_env_var
from langchain_replicate import ChatReplicate

os.environ["REPLICATE_API_TOKEN"] = userdata.get('REPLICATE_API_TOKEN')

model_path = "ibm-granite/granite-4.1-8b"
model = ChatReplicate(
    model=model_path,
    replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
    model_kwargs={
        "max_completion_tokens": 1000,
        "min_tokens": 100,
    },
)

print("✅ Granite model ready!")

✅ Granite model ready!


In [11]:
from docling.document_converter import DocumentConverter
from urllib.parse import urlparse

converter = DocumentConverter()

# Wikipedia page for 2024 Monaco Grand Prix - reliable and narrative
sources = [
    "https://en.wikipedia.org/wiki/2024_Monaco_Grand_Prix",
]

conversions = {
    source.split("/")[-1]: converter.convert(source=source).document
    for source in sources
}

print("✅ 2024 Monaco Grand Prix page loaded!")

✅ 2024 Monaco Grand Prix page loaded!


In [12]:
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.types.doc.document import TableItem
from langchain_core.documents import Document

doc_id = 0
texts = []

for source, docling_document in conversions.items():
    for chunk in HybridChunker(tokenizer=embeddings_tokenizer).chunk(docling_document):
        items = chunk.meta.doc_items
        if len(items) == 1 and isinstance(items[0], TableItem):
            continue
        text = chunk.text
        document = Document(
            page_content=text,
            metadata={
                "doc_id": (doc_id := doc_id + 1),
                "source": source,
            },
        )
        texts.append(document)

print(f"✅ {len(texts)} text chunks created from F1 race report")

✅ 17 text chunks created from F1 race report


In [13]:
from langchain_chroma import Chroma

vector_db = Chroma(embedding_function=embeddings_model)
ids = []
for doc in texts:
    ids += vector_db.add_documents([doc])

print(f"✅ {len(ids)} chunks added to vector database")

✅ 17 chunks added to vector database


In [14]:
from ibm_granite_community.langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("{input}")

combine_docs_chain = create_stuff_documents_chain(
    llm=model,
    prompt=prompt_template,
)
rag_chain = create_retrieval_chain(
    retriever=vector_db.as_retriever(),
    combine_docs_chain=combine_docs_chain,
)

print("✅ RAG chain ready!")

✅ RAG chain ready!


In [15]:
# Ask a question about the race — this is your Q&A feature working
question = "Who won the race and what were the key moments?"
output = rag_chain.invoke({"input": question})
print("🏎️ PitLane Explained Answer:")
print(output['answer'])

🏎️ PitLane Explained Answer:
Charles Leclerc won the 2024 Monaco Grand Prix. The key moments included:

1. **Red Flag on Lap 1**: The race was immediately halted on lap 1 due to a crash involving Sergio Pérez, Nico Hülkenberg, and Kevin Magnussen. This incident caused heavy damage to the barriers and spread debris across the first corners. Pérez's car incurred an estimated £2.5-3 million in repair costs.

2. **Impact on Qualifying Grid**: The red flag allowed for a grid reset, benefiting Carlos Sainz Jr., who had picked up a puncture during qualifying and dropped to sixteenth. Zhou Guanyu, who was behind the crash, slowed down to avoid further incidents.

3. **Strategic Adjustments**: The early red flag enabled drivers to swap to a second tyre compound during the red flag period, eliminating the need for pit stops for the rest of the race. Only Lewis Hamilton and Max Verstappen made further pit stops after gaining a significant lead.

4. **Race Outcome**: Leclerc maintained the lead fo

In [16]:
# FEATURE 2 - Race Summary Generator
# Generates a narrative summary of the race for casual fans

summary_question = """
Give me a short, engaging 3-paragraph summary of this race written
for someone who has never watched F1 before. Explain what happened,
why it was exciting, and what made the winner's victory special.
Use simple language and avoid technical jargon.
"""

summary_output = rag_chain.invoke({"input": summary_question})
print("🏁 RACE STORY:")
print("=" * 50)
print(summary_output['answer'])

🏁 RACE STORY:
The 2024 Monaco Grand Prix was a thrilling Formula 1 race held on the iconic streets of Monte Carlo, known for its tight, twisty track that tests drivers' precision and bravery. The event featured 20 drivers from various teams, each vying for the top spot on the starting grid. Charles Leclerc, driving for Ferrari, secured pole position with a blistering qualifying time of 1:10.270, setting the stage for an intense battle on race day.

As the race began, the cars zipped around the narrow circuit, navigating sharp corners and tight sections where overtaking was nearly impossible. The excitement peaked when several drivers, including Max Verstappen and Lando Norris, engaged in close, high-speed duels, showcasing their skill and nerve. The race was further electrified by unexpected incidents, such as collisions involving Sergio Pérez and Kevin Magnussen, which added drama and unpredictability to the proceedings.

Charles Leclerc's victory was particularly special because it m